# 5. A small neural material is still a density

**Goal:** replace the analytic density by an analytic baseline plus a tiny smooth
network, while keeping the FEM interface unchanged. All network weights are fixed
in this lesson; there is no training loop yet.

We choose a teaching model rather than a pretrained material:
$W=W_{\mathrm{NH}}+\alpha\det(F_p)\|z\|^2 f_\omega(z)$.
The Neo-Hookean baseline uses fixed $\lambda=2,\mu=3$ and $\alpha=0.2$.
This is a local demonstration, not a guarantee of global material stability.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
from litepgo import dof, energy, fem, solver
from litepgo.torch import TorchVolumetricMaterial, response
from common import make_geometry

torch.set_num_threads(1)
dtype = torch.float64

In [ ]:
class TorchNeoHookean(TorchVolumetricMaterial):
    num_parameters = 2

    @staticmethod
    def W(F, Fp, theta):
        Fe = F @ torch.linalg.inv(Fp)
        lam, mu = theta.unbind(-1)
        log_J = torch.linalg.det(Fe).log()
        psi = 0.5 * mu * (Fe.square().sum((-2, -1)) - 3) - mu * log_J + 0.5 * lam * log_J.square()
        W = torch.linalg.det(Fp) * psi
        return torch.where((lam >= 0) & (mu > 0), W, torch.full_like(W, float('nan')))

## Choose two invariant features

With $F_e=FF_p^{-1}$ and $J=\det F_e$, use
$z=(J^{-2/3}\|F_e\|^2-3,\log J)$. Both features vanish at rest, and rigid rotations
do not change them. A tiny 2–2–1 network computes

$$h(z)=\operatorname{softplus}(Az+b),\qquad f_\omega(z)=h(z)^T\omega.$$

Multiplying the correction by $\|z\|^2$ makes its value and first derivative zero
at rest. Unlike the advanced example, no origin-slope autograd correction is needed.
The two hidden units use fixed `A` and `b`; output weights will become explicit inputs
in lesson 6. Every operation is batched and reads only its row's weights.

In [ ]:
A = torch.tensor([[1.0, 0.5], [-0.7, 1.2]], dtype=dtype)
b = torch.tensor([0.2, -0.1], dtype=dtype)
alpha = 0.2

def invariants(Fe):
    J = torch.linalg.det(Fe)
    I1 = Fe.square().sum((-2, -1))
    return torch.stack((J.pow(-2 / 3) * I1 - 3, J.log()), dim=-1)

def neural_density(F, Fp, weights):
    Fe = F @ torch.linalg.inv(Fp)
    z = invariants(Fe)
    hidden = torch.nn.functional.softplus(z @ A.T + b)
    network = (hidden * weights).sum(-1)
    correction = alpha * z.square().sum(-1) * network
    physical = F.new_tensor([2., 3.]).expand(len(F), 2)
    return TorchNeoHookean.W(F, Fp, physical) + torch.linalg.det(Fp) * correction
weights = torch.tensor([[0.6, 0.4]], dtype=dtype)
F = torch.diag(torch.tensor([1.1, 1., 1.], dtype=dtype))[None].requires_grad_()
Fp = torch.eye(3, dtype=dtype)[None]
W = neural_density(F, Fp, weights)
P = torch.autograd.grad(W.sum(), F)[0]
print('neural density:', W.item())
print('stress P11:', P[0, 0, 0].item())

## Check reference state and a rigid rotation

The correction should not create rest energy or rest stress. A rotated deformation
should have the same energy. These are physical checks on the density, separate from
checking the mechanics of autograd.

In [ ]:
rest = torch.eye(3, dtype=dtype)[None].requires_grad_()
Wr = neural_density(rest, Fp, weights)
Pr = torch.autograd.grad(Wr.sum(), rest)[0]
assert Wr.abs().max() < 1e-12 and Pr.abs().max() < 1e-12
angle = torch.tensor(0.4, dtype=dtype)
Q = torch.tensor([[angle.cos(), -angle.sin(), 0.], [angle.sin(), angle.cos(), 0.], [0., 0., 1.]], dtype=dtype)
torch.testing.assert_close(neural_density(Q @ F, Fp, weights), W, rtol=1e-11, atol=1e-12)
print('reference state and rotation checks passed')

## Use the same material adapter

A fixed network has no active material parameters: `num_parameters = 0` and each
`theta` row is empty. Geometry, integration, and stiffness assembly still work.
The local tangent check below verifies a Hessian–direction product without writing
out all 81 tangent entries.

In [ ]:
class FixedNeuralMaterial(TorchVolumetricMaterial):
    num_parameters = 0

    @staticmethod
    def W(F, Fp, theta):
        return neural_density(F, Fp, weights.expand(len(F), -1))

f = F.detach().numpy()
fp = Fp.numpy()
empty = np.empty((1, 0))
out = FixedNeuralMaterial.eval(f, fp, empty, outputs=('W', 'dW_dF', 'd2W_dF2'))
direction = np.arange(9).reshape(1, 3, 3) / 10
h = 1e-5
plus = FixedNeuralMaterial.eval(f + h * direction, fp, empty, outputs=('dW_dF',))['dW_dF']
minus = FixedNeuralMaterial.eval(f - h * direction, fp, empty, outputs=('dW_dF',))['dW_dF']
actual = (out['d2W_dF2'].reshape(9, 9) @ direction.ravel()).reshape(1, 3, 3)
np.testing.assert_allclose(actual, (plus - minus) / (2*h), rtol=1e-6, atol=1e-9)
X, topology, mapping, geometry, plastic = make_geometry()
empty_fields = [energy.AffineResponse(np.zeros((0, 3)), np.empty(0))] * topology.num_cells
potential = fem.VolumetricFemEnergy[FixedNeuralMaterial, fem.TetP1Basis, fem.TetCentroidQuadrature](
    mapping, geometry, plastic, empty_fields)
u = (X * np.array([.08, -.02, .04])).ravel()
E, g, K = potential.eval(u, gradient=True, hessian=energy.HessianKind.Exact)
assert np.isfinite(E) and np.isfinite(g).all() and np.isfinite(K.data).all()
print('mesh energy:', E, 'stiffness shape:', K.shape)

## Compare the loading curves

The correction changes the constitutive behavior, not the FEM interface. The network
is deliberately tiny so you can see every operation that controls the density.

In [ ]:
stretch = torch.linspace(.8, 1.2, 61, dtype=dtype)
path = torch.eye(3, dtype=dtype).repeat(len(stretch), 1, 1)
path[:, 0, 0] = stretch
plastic_path = torch.eye(3, dtype=dtype).expand(len(stretch), 3, 3)
base = TorchNeoHookean.W(path, plastic_path, path.new_tensor([2., 3.]).expand(len(stretch), 2))
learned = neural_density(path, plastic_path, weights.expand(len(stretch), 2))
fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
ax.plot(stretch.numpy(), base.numpy(), label='Analytic baseline')
ax.plot(stretch.numpy(), learned.numpy(), '--', label='Baseline + neural correction')
ax.set(xlabel='Stretch s', ylabel='Energy density')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## Try it

Change the fixed output weights. Verify that rest energy and stress stay zero, then
look at the loading curve and tangent. Smoothness does not imply a positive definite
tangent everywhere; large changes may produce an unstable material.

**Checkpoint:** a network material obeys exactly the same density/derivative contract.
Next: turn its output weights into the explicit parameters of an equilibrium fit.